# <center> Deep Learning Coursework </center>
## <center> Audio Text Transformer Implementation </center>

Explaination of Audio Text Transformer

Imports required

In [ ]:
import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import transforms

## Optimiser comparison: Adam vs AdamW
Same seeds, lr, epochs and architecture; only the optimiser changes. Efficiency is measured as the cost of reaching a fixed target accuracy, not just final accuracy.

In [ ]:
import sys, shlex, json
from pathlib import Path

# ---- experiment config -------------------------------------------------
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 0.01          # AdamW only
SEEDS = [42, 43, 44]         # >1 seed so we can tell real gains from noise; drop to [42] if slow
CONFIGS = {
    "adam":  ["--optimizer", "adam"],
    "adamw": ["--optimizer", "adamw", "--weight-decay", str(WEIGHT_DECAY)],
}

# Each run is a separate process (clean state), saved to ./<name>_s<seed>/
for name, extra in CONFIGS.items():
    for seed in SEEDS:
        run = f"{name}_s{seed}"
        if (Path(run) / "metrics.json").exists():
            print(f"[skip] {run} already done (delete the folder to re-run)")
            continue
        cmd = [sys.executable, "audio-text_transformer.py",
               "--learning-rate", str(LR), "--epochs", str(EPOCHS),
               "--seed", str(seed), "--model-name", run,
               "--val-frequency", "1",          # validate every epoch -> needed for time-to-accuracy
               "--print-frequency", "1000000",  # silence per-step prints
               *extra]
        print("RUN:", shlex.join(cmd))
        get_ipython().system(shlex.join(cmd))

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def load(run):
    with open(Path(run) / "metrics.json") as f:
        return json.load(f)

runs = {name: [load(f"{name}_s{s}") for s in SEEDS] for name in CONFIGS}

# Target accuracy for "time/energy/FLOPs to reach X": the worst baseline run's best val accuracy,
# so every baseline run is guaranteed to reach it.
TARGET = min(max(m["history"]["val_accuracy"]) for m in runs["adam"])
print(f"Target val accuracy for efficiency comparison: {TARGET*100:.2f}%")

def to_target(m, target=TARGET):
    """Cost of training until val accuracy first reaches `target` (None if never)."""
    h = m["history"]
    cum_t = np.cumsum(h["epoch_train_time"])
    e = h["epoch_energy_j"]
    cum_e = np.cumsum(e) if all(x is not None for x in e) else None
    for ep, acc in zip(h["val_epochs"], h["val_accuracy"]):
        if acc >= target:
            flops = (3 * m["fwd_flops_per_sample"] * m["samples_per_epoch"] * ep
                     if m["fwd_flops_per_sample"] else np.nan)  # train ~ 3x forward
            return dict(epochs=ep, seconds=cum_t[ep-1], energy_j=cum_e[ep-1] if cum_e is not None else np.nan,
                        train_tflops=flops / 1e12)
    return dict(epochs=np.nan, seconds=np.nan, energy_j=np.nan, train_tflops=np.nan)

rows = []
for name, ms in runs.items():
    for m in ms:
        h = m["history"]
        rows.append(dict(
            optimiser=name, seed=m["seed"],
            final_val_acc=h["val_accuracy"][-1] * 100,
            best_val_acc=max(h["val_accuracy"]) * 100,
            total_train_s=m["total_train_seconds"],
            peak_gpu_mb=m["peak_gpu_mem_mb"],
            params=m["trainable_params"],
            **{f"to_target_{k}": v for k, v in to_target(m).items()},
        ))
df = pd.DataFrame(rows)
display(df.round(3))

summary = df.drop(columns="seed").groupby("optimiser").agg(["mean", "std"]).round(3)
display(summary.T)

### Accuracy vs compute

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for name, ms in runs.items():
    acc = np.array([m["history"]["val_accuracy"] for m in ms]) * 100
    ep = ms[0]["history"]["val_epochs"]
    cum_t = np.mean([np.cumsum(m["history"]["epoch_train_time"]) for m in ms], axis=0)
    ax[0].errorbar(ep, acc.mean(0), acc.std(0), marker="o", capsize=3, label=name)
    ax[1].plot(cum_t, acc.mean(0), marker="o", label=name)
    loss = np.array([m["history"]["val_loss"] for m in ms])
    ax[2].errorbar(ep, loss.mean(0), loss.std(0), marker="o", capsize=3, label=name)
ax[0].set(xlabel="Epoch", ylabel="Val accuracy (%)", title="Accuracy vs epoch (mean ± std)")
ax[1].set(xlabel="Cumulative training time (s)", ylabel="Val accuracy (%)", title="Accuracy vs compute")
ax[2].set(xlabel="Epoch", ylabel="Val loss", title="Val loss vs epoch")
ax[0].axhline(TARGET * 100, ls="--", c="gray", lw=0.8)
for a in ax: a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig("optimiser_comparison.png", dpi=300, bbox_inches="tight"); plt.show()

### Re-evaluating saved weights

In [ ]:
# Evaluate saved weights. Use "val" while developing; only use "test" for the final reported numbers.
import importlib.util
import torch
from torch.utils.data import DataLoader

spec = importlib.util.spec_from_file_location("att", "audio-text_transformer.py")
att = importlib.util.module_from_spec(spec); spec.loader.exec_module(att)

DEVICE = att.DEVICE
DATASET_ROOT = Path("/data/COMS30017")

@torch.no_grad()
def evaluate(weights_path, split="val", batch_size=128):
    audio, text, labels = att.load_features(DATASET_ROOT, split)
    ds = att.AudioTextDataset(audio, text, labels, training=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    a0, t0, _ = ds[0]
    model = att.AudioTextTransformer(audio_dim=a0.shape[-1], text_dim=t0.shape[-1],
                                     d_model=att.MODEL_DIM, num_heads=4,
                                     mlp_dim=att.TRANSFORMER_MLP_DIM, dropout=att.DROPOUT)
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
    model.to(DEVICE).eval()
    crit = torch.nn.CrossEntropyLoss()
    loss, preds, labs = 0.0, [], []
    for a, t, y in loader:
        logits = model(a.to(DEVICE), t.to(DEVICE))
        loss += crit(logits, y.to(DEVICE)).item()
        preds += logits.argmax(-1).cpu().tolist(); labs += y.tolist()
    acc = float(np.mean(np.array(preds) == np.array(labs)))
    return loss / len(loader), acc, len(labs)

for name in CONFIGS:
    for s in SEEDS:
        l, a, n = evaluate(Path(f"{name}_s{s}") / "transformer_weights.pth", "val")
        print(f"{name:6s} seed {s}: loss {l:.5f}  acc {a*100:.2f}%  (n={n})")